In [1]:
import pandas as pd
import re

from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()

print("Libraries Loaded")

StatementMeta(, b2e9999c-15f9-45b5-89dc-81cf96aa50e7, 3, Finished, Available, Finished, False)

Libraries Loaded


In [39]:
required_categories = [

    "FBSN",
    "FMSN",
    "FSIN",
    "FSMB",

    "HBSF",
    "HBWP",
    "HBWF",

    "HDOF",
    "HDOM",
    "HFCN",

    "HSKM",
    "HSYF"
]
## Change by Sourav
required_kpis = [
    "Premium Index",
    "Perceived Price Index",
    "Perceived Worth Index",

    "Better Quality Index",
    "Attractive Pack",
    "Recycled Packaging",
    "SLP - Absolute",
    "SLP Index",
    "SLP - Profile",
    "Recycled Packaging",
    "Recycled Packaging Index",
    "Behave Socially Responsible",
    "Behave Socially Responsible Index",
    "Environmentally Responsible",
    "Environmentally Responsible Index",
    "Packaging Better For Environment",
    "Packaging Better For Environment Index"

]

pricing_metrics = [

    "Premium Index",
    "Perceived Price Index",
    "Perceived Worth Index"

]
## Change by Sourav
sustainibility_metrics = [
    "SLP - Absolute",
    "SLP Index",
    "SLP - Profile",
    "Recycled Packaging",
    "Recycled Packaging Index",
    "Behave Socially Responsible",
    "Behave Socially Responsible Index",
    "Environmentally Responsible",
    "Environmentally Responsible Index",
    "Packaging Better For Environment",
    "Packaging Better For Environment Index"
]

StatementMeta(, b2e9999c-15f9-45b5-89dc-81cf96aa50e7, 41, Finished, Available, Finished, False)

In [41]:
import re

files = mssparkutils.fs.ls("Files/6P")

pattern = r"^Q[1-4]\s\d{4}\.xlsx$"

quarterly_files = sorted(
    [
        f.name
        for f in files
        if re.match(pattern, f.name)
    ]
)

print("Files Found:", len(quarterly_files))

for file in quarterly_files:
    print(file)

StatementMeta(, b2e9999c-15f9-45b5-89dc-81cf96aa50e7, 43, Finished, Available, Finished, False)

Files Found: 13
Q1 2023.xlsx
Q1 2024.xlsx
Q1 2025.xlsx
Q1 2026.xlsx
Q2 2023.xlsx
Q2 2024.xlsx
Q2 2025.xlsx
Q3 2023.xlsx
Q3 2024.xlsx
Q3 2025.xlsx
Q4 2023.xlsx
Q4 2024.xlsx
Q4 2025.xlsx


In [42]:
combined_dfs = []

for file in quarterly_files:

    path = f"/lakehouse/default/Files/6P/{file}"

    df = pd.read_excel(path)

    # USA Filter

    df = df[
        df["Country"]
        .str.upper()
        .str.strip()
        ==
        "UNITED STATES OF AMERICA"
    ]

    # Category Filter

    df = df[
        df["Category Code"]
        .isin(required_categories)
    ]
    # KPI Filter

    df = df[
        df["Measure/KPI"]
        .isin(required_kpis)
    ]
    

    # Metric Group

    df["Metric_Group"] = df["Measure/KPI"].apply(

        lambda x:

        "Pricing"

        if x in pricing_metrics 
        
        else "Sustainibility"
        
        if x in sustainibility_metrics
        
        else "Additional UBS Metrics"
    )

    df["Source_File"] = file

    combined_dfs.append(df)

    print(
        file,
        "-->",
        df.shape
    )


StatementMeta(, b2e9999c-15f9-45b5-89dc-81cf96aa50e7, 44, Finished, Available, Finished, False)

Q1 2023.xlsx --> (1221, 11)
Q1 2024.xlsx --> (1461, 11)
Q1 2025.xlsx --> (1524, 11)
Q1 2026.xlsx --> (1689, 11)
Q2 2023.xlsx --> (1136, 11)
Q2 2024.xlsx --> (1424, 11)
Q2 2025.xlsx --> (1878, 11)
Q3 2023.xlsx --> (1434, 11)
Q3 2024.xlsx --> (1483, 11)
Q3 2025.xlsx --> (1629, 11)
Q4 2023.xlsx --> (1436, 11)
Q4 2024.xlsx --> (1438, 11)
Q4 2025.xlsx --> (1593, 11)


In [44]:
combined_pdf = pd.concat(
    combined_dfs,
    ignore_index=True
)

# Round MAT Values to 2 decimal places
combined_pdf["MAT Values"] = combined_pdf["MAT Values"].round(2)
## Change by Sourav
combined_pdf["Measure/KPI"] = combined_pdf["Measure/KPI"].str.replace("SLP - Absolute","SLP Absolute").replace("SLP - Profile","SLP Profile")
print("Combined Shape:", combined_pdf.shape)

StatementMeta(, b2e9999c-15f9-45b5-89dc-81cf96aa50e7, 46, Finished, Available, Finished, False)

Combined Shape: (19346, 11)


In [46]:
import re

combined_pdf.columns = [

    re.sub(
        r'[^A-Za-z0-9_]',
        '_',
        str(col).strip()
    )

    for col in combined_pdf.columns
]

print(combined_pdf.columns.tolist())

StatementMeta(, b2e9999c-15f9-45b5-89dc-81cf96aa50e7, 48, Finished, Available, Finished, False)

['Category_Code', 'Kantar_Category_Name', 'Country', 'Category_Name', 'Brand_Name', 'Measure_KPI', 'MAT_QUARTER', 'MAT_Values', 'MAT_Sig', 'Metric_Group', 'Source_File']


In [47]:
print("Rows:", len(combined_pdf))

print("\nYears / Quarters")

print(
    combined_pdf["MAT_QUARTER"]
    .value_counts()
)

StatementMeta(, b2e9999c-15f9-45b5-89dc-81cf96aa50e7, 49, Finished, Available, Finished, False)

Rows: 19346

Years / Quarters
MAT_QUARTER
MAT (Q2 '25)    1878
MAT (Q1 '26)    1689
MAT (Q3 '25)    1629
MAT (Q4 '25)    1593
MAT (Q1 '25)    1524
MAT (Q3 '24)    1483
MAT (Q1 '24)    1461
MAT (Q4 '24)    1438
MAT (Q4 '23)    1436
MAT (Q3 '23)    1434
MAT (Q2 '24)    1424
MAT (Q1 '23)    1221
MAT (Q2 '23)    1136
Name: count, dtype: int64


In [48]:
print("\nMetric Groups")

print(
    combined_pdf["Metric_Group"]
    .value_counts()
)

StatementMeta(, b2e9999c-15f9-45b5-89dc-81cf96aa50e7, 50, Finished, Available, Finished, False)


Metric Groups
Metric_Group
Sustainibility            12490
Pricing                    4060
Additional UBS Metrics     2796
Name: count, dtype: int64


In [49]:
print("\nKPIs")

print(
    combined_pdf["Measure_KPI"]
    .value_counts()
)

StatementMeta(, b2e9999c-15f9-45b5-89dc-81cf96aa50e7, 51, Finished, Available, Finished, False)


KPIs
Measure_KPI
Better Quality Index                 1416
Attractive Pack                      1380
SLP Profile                          1380
Environmentally Responsible          1379
Environmentally Responsible Index    1379
Perceived Worth Index                1379
Recycled Packaging                   1379
Recycled Packaging Index             1379
SLP Absolute                         1379
SLP Index                            1379
Premium Index                        1378
Perceived Price Index                1303
Behave Socially Responsible          1184
Behave Socially Responsible Index    1184
Packaging Better For Environment      468
Name: count, dtype: int64


In [50]:
spark_df = spark.createDataFrame(
    combined_pdf
)

print(
    spark_df.count()
)

StatementMeta(, b2e9999c-15f9-45b5-89dc-81cf96aa50e7, 52, Finished, Available, Finished, False)

19346


In [51]:
spark_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(
        "PricingAndUBSMetrics_table"
    )

print(
    "PricingAndUBSMetrics_table Saved Successfully"
)

StatementMeta(, b2e9999c-15f9-45b5-89dc-81cf96aa50e7, 53, Finished, Available, Finished, False)

PricingAndUBSMetrics_table Saved Successfully


In [52]:
display(

    spark.sql("""

    SELECT

        MAT_QUARTER,
        Metric_Group,

        COUNT(*) AS Rows

    FROM PricingAndUBSMetrics_table

    GROUP BY

        MAT_QUARTER,
        Metric_Group

    ORDER BY

        MAT_QUARTER,
        Metric_Group

    """)

)

StatementMeta(, b2e9999c-15f9-45b5-89dc-81cf96aa50e7, 54, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 72a835a7-2d79-4130-b1da-792c9977da1c)

In [53]:
question_brands_df = (
    spark.table("QuestionTable")
         .select("Brands")
         .distinct()
         .orderBy("Brands")
         .toPandas()
)

print("QuestionTable Brand Count:", len(question_brands_df))

display(question_brands_df)

StatementMeta(, b2e9999c-15f9-45b5-89dc-81cf96aa50e7, 55, Finished, Available, Finished, False)

QuestionTable Brand Count: 32


SynapseWidget(Synapse.DataFrame, 66c558d5-ec3f-4158-8c37-89395aec7ad4)

In [54]:
pricing_brands_df = (
    spark.table("pricingandubsmetrics_table")
         .select("Brand_Name")
         .distinct()
         .orderBy("Brand_Name")
         .toPandas()
)

print("PricingAndUBSMetrics Brand Count:", len(pricing_brands_df))

display(pricing_brands_df)

StatementMeta(, b2e9999c-15f9-45b5-89dc-81cf96aa50e7, 56, Finished, Available, Finished, False)

PricingAndUBSMetrics Brand Count: 108


SynapseWidget(Synapse.DataFrame, 9bfa8596-3237-437f-be84-1f90944aea37)

In [55]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when


# Create Spark session
spark = SparkSession.builder.getOrCreate()


# Read tables
my_df = spark.table("pricingandubsmetrics_table")
question_df = spark.table("questiontable")


# Get distinct brands
my_brand_df = my_df.select("Brand_Name").distinct()

question_brand_df = question_df.select("Brands").distinct()


# Compare using left join
check_df = my_brand_df.join(
    question_brand_df,
    my_brand_df.Brand_Name == question_brand_df.Brands,
    "left"
)


# Add match status
result_df = check_df.withColumn(
    "match_status",
    when(
        col("Brands").isNotNull(),
        "Matched"
    ).otherwise("Not Matched")
)


# Filter only Not Matched brands
not_matched_df = result_df.filter(
    col("match_status") == "Not Matched"
)


# Keep only required column
final_df = not_matched_df.select(
    "Brand_Name",
    "match_status"
)


# Convert to pandas
final_pandas_df = final_df.toPandas()


# Display in Fabric with download option
display(final_pandas_df)

StatementMeta(, b2e9999c-15f9-45b5-89dc-81cf96aa50e7, 57, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 30e41343-7d81-43e9-b702-a024a74f7607)

In [53]:
# import pandas as pd
# import os
# import re

# from pyspark.sql import SparkSession

# spark = SparkSession.builder.getOrCreate()

# print("Libraries Loaded")

StatementMeta(, b9cb5481-32f9-4282-a33a-f57a2abd1f52, 55, Finished, Available, Finished, False)

In [54]:
# folder_path = "/lakehouse/default/Files/6P"

# required_categories = [

#     "FBSN",
#     "FMSN",
#     "FSIN",
#     "FSMB",

#     "HBSF",
#     "HBWP",
#     "HBWF",

#     "HDOF",
#     "HDOM",
#     "HFCN",

#     "HSKM",
#     "HSYF"
# ]

# required_kpis = [

#     "Premium Index",
#     "Perceived Price Index",
#     "Perceived Worth Index",

#     "Better Quality Index",
#     "Attractive Pack",
#     "Recycled Packaging"
# ]

# print("Filters Created")

StatementMeta(, b9cb5481-32f9-4282-a33a-f57a2abd1f52, 56, Finished, Available, Finished, False)

In [55]:
# all_filtered_dfs = []

# for file in sorted(os.listdir(folder_path)):

#     # Skip non Excel files
#     if not file.endswith(".xlsx"):
#         continue

#     # Skip generated quarter files
#     # Example:
#     # Q1 2023.xlsx
#     # Q2 2024.xlsx
#     # Q3 2025.xlsx

#     if re.match(r"^Q[1-4]\s\d{4}\.xlsx$", file):
#         continue

#     file_path = os.path.join(
#         folder_path,
#         file
#     )

#     xls = pd.ExcelFile(file_path)

#     print(f"\nProcessing: {file}")
#     print(f"Available Sheets: {xls.sheet_names}")

#     # Find 6P sheet

#     sixp_sheets = [

#         s
#         for s in xls.sheet_names

#         if "6P" in s
#     ]

#     # Safety Check

#     if len(sixp_sheets) == 0:

#         print(
#             f"Skipping {file} - No 6P sheet found"
#         )

#         continue

#     sheet_name = sixp_sheets[0]

#     df = pd.read_excel(
#         file_path,
#         sheet_name=sheet_name
#     )

#     # Clean Column Names

#     df.columns = [

#         re.sub(
#             r"[^A-Za-z0-9_]",
#             "_",
#             str(col).strip()
#         )

#         for col in df.columns
#     ]

#     # USA Filter

#     df = df[

#         df["Country"]
#         .astype(str)
#         .str.upper()
#         .str.strip()

#         ==

#         "UNITED STATES OF AMERICA"
#     ]

#     # Category Filter

#     df = df[

#         df["Category_Code"]
#         .isin(required_categories)
#     ]

#     # KPI Filter

#     df = df[

#         df["Measure_KPI"]
#         .isin(required_kpis)
#     ]

#     # Source File

#     df["Source_File"] = file

#     all_filtered_dfs.append(df)

#     print(
#         f"{file} --> {df.shape}"
#     )

# print(
#     "\nTotal Files Processed:",
#     len(all_filtered_dfs)
# )

StatementMeta(, b9cb5481-32f9-4282-a33a-f57a2abd1f52, 57, Finished, Available, Finished, False)

In [56]:
# standardized_dfs = []

# for df in all_filtered_dfs:

#     temp_df = df.copy()

#     if "Kantar_Category_Name" not in temp_df.columns:

#         temp_df["Kantar_Category_Name"] = temp_df[
#             "Unilever_Category_Name"
#         ]

#     standardized_dfs.append(temp_df)

# print(
#     "Total Standardized Files:",
#     len(standardized_dfs)
# )

StatementMeta(, b9cb5481-32f9-4282-a33a-f57a2abd1f52, 58, Finished, Available, Finished, False)

In [57]:
# pricing_metrics = [

#     "Premium Index",
#     "Perceived Price Index",
#     "Perceived Worth Index"
# ]

# for df in standardized_dfs:

#     df["Metric_Group"] = df[
#         "Measure_KPI"
#     ].apply(

#         lambda x:

#         "Pricing"

#         if x in pricing_metrics

#         else "Additional UBS Metrics"
#     )

# print("Metric Group Created")

StatementMeta(, b9cb5481-32f9-4282-a33a-f57a2abd1f52, 59, Finished, Available, Finished, False)

In [58]:
# def transform_file_current_mat(df):

#     base_cols = [

#         "Period_Ending_Date",

#         "Category_Code",
#         "Category_Name",
#         "Kantar_Category_Name",

#         "Country",
#         "Business_Group",

#         "Manufacturer",
#         "Brand_Name",

#         "Metric_Group",
#         "Measure_KPI",

#         "Source_File"
#     ]

#     temp = df[base_cols].copy()

#     # Current MAT Column

#     current_mat_col = [

#         c for c in df.columns

#         if c.startswith("MAT_")
#         and "__MAT_" in c
#         and "__MAT_1_" not in c
#         and "__MAT_2_" not in c
#         and "Sig" not in c
#     ][0]

#     # Current MAT Sig Column

#     current_sig_col = [

#         c for c in df.columns

#         if "__MAT_Sig_" in c
#         and "__MAT_1_Sig_" not in c
#         and "__MAT_2_Sig_" not in c
#     ][0]

#     # Quarter + Year

#     match = re.search(
#         r"MAT_(Q\d)_(\d{4})",
#         current_mat_col
#     )

#     temp["Quarter"] = match.group(1)

#     temp["Year"] = int(
#         match.group(2)
#     )

#     temp["Value"] = df[current_mat_col]

#     temp["Sig"] = df[current_sig_col]

#     return temp

StatementMeta(, b9cb5481-32f9-4282-a33a-f57a2abd1f52, 60, Finished, Available, Finished, False)

In [59]:
# final_history_df = pd.concat(

#     [

#         transform_file_current_mat(df)

#         for df in standardized_dfs

#     ],

#     ignore_index=True
# )

# print(
#     final_history_df.shape
# )

StatementMeta(, b9cb5481-32f9-4282-a33a-f57a2abd1f52, 61, Finished, Available, Finished, False)

In [60]:
# final_history_df["Period_Ending_Date"] = pd.to_datetime(
#     final_history_df["Period_Ending_Date"]
# ).dt.date

# final_history_df = final_history_df.drop_duplicates()

# print(
#     final_history_df.shape
# )

StatementMeta(, b9cb5481-32f9-4282-a33a-f57a2abd1f52, 62, Finished, Available, Finished, False)

In [61]:
# print(
#     final_history_df["Source_File"]
#     .nunique()
# )

# print(
#     final_history_df["Year"]
#     .value_counts()
#     .sort_index()
# )

# print(
#     final_history_df["Metric_Group"]
#     .value_counts()
# )

StatementMeta(, b9cb5481-32f9-4282-a33a-f57a2abd1f52, 63, Finished, Available, Finished, False)

In [62]:
# spark_final_df = spark.createDataFrame(
#     final_history_df
# )

# print(
#     spark_final_df.count()
# )

StatementMeta(, b9cb5481-32f9-4282-a33a-f57a2abd1f52, 64, Finished, Available, Finished, False)

In [63]:
# spark_final_df.write \
#     .format("delta") \
#     .mode("overwrite") \
#     .option("overwriteSchema", "true") \
#     .saveAsTable(
#         "PricingAndUBSMetrics_CurrentMAT"
#     )

# print(
#     "PricingAndUBSMetrics_CurrentMAT Saved Successfully"
# )

StatementMeta(, b9cb5481-32f9-4282-a33a-f57a2abd1f52, 65, Finished, Available, Finished, False)

In [64]:
# display(

#     spark.sql("""

#     SELECT *

#     FROM PricingAndUBSMetrics_CurrentMAT

#     ORDER BY

#         Year,
#         Quarter,
#         Metric_Group,
#         Brand_Name

#     """)

# )

StatementMeta(, b9cb5481-32f9-4282-a33a-f57a2abd1f52, 66, Finished, Available, Finished, False)

In [65]:
# brand_table1 = pd.DataFrame(
#     sorted(
#         combined_pdf["Brand_Name"]
#         .dropna()
#         .astype(str)
#         .str.strip()
#         .unique()
#     ),
#     columns=["Brand_Name"]
# )

# print("Table 1 Brand Count:", len(brand_table1))

# display(brand_table1)

StatementMeta(, b9cb5481-32f9-4282-a33a-f57a2abd1f52, 67, Finished, Available, Finished, False)

In [66]:
# question_brands = (
#     spark.table("QuestionTable")
#          .select("Brands")
#          .distinct()
#          .orderBy("Brands")
# )

# display(question_brands)

StatementMeta(, b9cb5481-32f9-4282-a33a-f57a2abd1f52, 68, Finished, Available, Finished, False)

In [67]:
# question_brands_df = (
#     spark.table("QuestionTable")
#          .select("Brands")
#          .distinct()
#          .orderBy("Brands")
#          .toPandas()
# )

# print("Question Table Brand Count:", len(question_brands_df))

# display(question_brands_df)

StatementMeta(, b9cb5481-32f9-4282-a33a-f57a2abd1f52, 69, Finished, Available, Finished, False)

In [68]:
# brand_table1 = (
#     spark.table("pricingandubsmetrics_table")
#          .select("Brand_Name")
#          .distinct()
#          .orderBy("Brand_Name")
#          .toPandas()
# )

# print("Our Table Brand Count:", len(brand_table1))
# display(brand_table1)

StatementMeta(, b9cb5481-32f9-4282-a33a-f57a2abd1f52, 70, Finished, Available, Finished, False)

In [69]:
# question_brands = set(
#     question_brands_df["Brands"]
#     .dropna()
#     .astype(str)
#     .str.strip()
# )

# our_brands = set(
#     brand_table1["Brand_Name"]
#     .dropna()
#     .astype(str)
#     .str.strip()
# )

# missing_in_our_table = sorted(question_brands - our_brands)
# extra_in_our_table = sorted(our_brands - question_brands)

# print("Missing in Our Table:", len(missing_in_our_table))
# print("=" * 50)

# for brand in missing_in_our_table:
#     print(brand)

# print("\nExtra in Our Table:", len(extra_in_our_table))
# print("=" * 50)

# for brand in extra_in_our_table:
#     print(brand)

StatementMeta(, b9cb5481-32f9-4282-a33a-f57a2abd1f52, 71, Finished, Available, Finished, False)

In [70]:
# brand_mapping = {

#     "Annie’s": "Annies",
#     "Annie’s Macaroni & Cheese": "Annies Macaroni & Cheese",

#     "Arm & Hammer": "Arm & Hammer [Parent]",

#     "Ben’s Original": "Ben's Original",

#     "Caress": "Caress (Unilever)",

#     "Dove Men+Care": "Dove Men + Care",

#     "Duke’s": "Dukes",

#     "Earth Breeze": "EarthBreeze",

#     "Frank’s RedHot": "Franks RedHot",

#     "French’s (McCormick)": "French's",

#     "Fructis": "Garnier Fructis",

#     "Whole Blends": "Garnier Whole Blends",

#     "Head & Shoulders": "Head & Shoulders (H & S)",

#     "Hellmann’s": "Hellmanns",

#     "Hidden Valley Ranch": "Hidden Valley",

#     "Lady Speed Stick": "Lady Speedstick (Parent)",

#     "L’Oreal": "LOreal",

#     "Lush": "Lush (Unilever)",

#     "Schmidt’s": "Schmidts",

#     "Seventh Generation": "Seventh Generation (Unilever)",

#     "Shea Moisture": "Shea Moisture (Unilever)",

#     "Sir Kensington’s": "Sir Kensingtons",

#     "Sweet Baby Ray’s": "Sweet Baby Rays",

#     "TRESemme": "Tresemme",

#     "Tom’s of Maine": "Toms of Maine",

#     "Wyler’s": "Wylers",

#     "Zest (P&G)": "Zest",

#     "Nivea Men": "Nivea for Men",

#     "Ogx": "OGX",

#     "Pasta Roni": "Rice-A-Roni/Pasta Roni",

#     "Yardley London": "Yardley",

#     "Method Men Every Man Jack and Cremo": "Method Men",

#     "Minute": "Minute Rice",

#     "Kraft (Kraft)": "Kraft"
# }

StatementMeta(, b9cb5481-32f9-4282-a33a-f57a2abd1f52, 72, Finished, Available, Finished, False)

In [71]:
# combined_pdf["Brand_Name"] = (
#     combined_pdf["Brand_Name"]
#     .replace(brand_mapping)
# )

StatementMeta(, b9cb5481-32f9-4282-a33a-f57a2abd1f52, 73, Finished, Available, Finished, False)

In [72]:
# pricing_df = spark.table("pricingandubsmetrics_table").toPandas()

# pricing_df["Brand_Name"] = (
#     pricing_df["Brand_Name"]
#     .replace(brand_mapping)
# )

StatementMeta(, b9cb5481-32f9-4282-a33a-f57a2abd1f52, 74, Finished, Available, Finished, False)

In [73]:
# for old_name in brand_mapping.keys():
#     cnt = len(
#         pricing_df[
#             pricing_df["Brand_Name"] == old_name
#         ]
#     )

#     if cnt > 0:
#         print(old_name, "->", cnt)

StatementMeta(, b9cb5481-32f9-4282-a33a-f57a2abd1f52, 75, Finished, Available, Finished, False)

In [74]:
# spark.createDataFrame(pricing_df) \
#      .write \
#      .mode("overwrite") \
#      .option("overwriteSchema","true") \
#      .saveAsTable("pricingandubsmetrics_table")

StatementMeta(, b9cb5481-32f9-4282-a33a-f57a2abd1f52, 76, Finished, Available, Finished, False)